In [1]:
from __future__ import annotations

import sys
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
DATASETS = ["drd2", "hiv", "kdr", "sol"]
TASK = "hi"

def find_project_root(start: Path | None = None) -> Path:
    """
    Walk upward until the project root is found.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start
    while current != current.parent:
        if (
            (current / "data").exists()
            and (current / "utils").exists()
            and (current / "results").exists()
        ):
            return current
        current = current.parent

    raise RuntimeError("Could not find project root containing data/, utils/, and results/.")

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.fingerprints import compute_fingerprints

OUT_ROOT = PROJECT_ROOT / "results" / "results_classifier_shift_test" / TASK
FIG_ROOT = OUT_ROOT / "figures"

# Root feature cache directory
FEATURE_CACHE_ROOT = PROJECT_ROOT / "features" / "classifier_shift_test" / TASK

for d in (OUT_ROOT, FIG_ROOT, FEATURE_CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

FINGERPRINTS = ["ecfp4", "maccs", "rdkit_desc"]

TOPK_VALUES = [10, 20, 50]

N_CV_SPLITS = 5

print(f"Datasets     : {DATASETS}")
print(f"Task         : {TASK}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {OUT_ROOT}")
print(f"Figure root  : {FIG_ROOT}")
print(f"Feature cache: {FEATURE_CACHE_ROOT}")

Datasets     : ['drd2', 'hiv', 'kdr', 'sol']
Task         : hi
Project root : /home/f.capria/drug-discovery-lohi
Output root  : /home/f.capria/drug-discovery-lohi/results/results_classifier_shift_test/hi
Figure root  : /home/f.capria/drug-discovery-lohi/results/results_classifier_shift_test/hi/figures
Feature cache: /home/f.capria/drug-discovery-lohi/features/classifier_shift_test/hi


In [3]:
def get_dataset_paths(dataset: str) -> dict:
    """
    Return all input/output paths for one Hi dataset.
    """
    data_dir = PROJECT_ROOT / "data" / TASK / dataset

    binding_results_dir = (
        PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / dataset
    )

    out_dir = OUT_ROOT / dataset
    fig_dir = out_dir / "figures"
    feat_dir = out_dir / "discriminator_features"
    feature_cache_dir = FEATURE_CACHE_ROOT / dataset

    for d in (out_dir, fig_dir, feat_dir, feature_cache_dir):
        d.mkdir(parents=True, exist_ok=True)

    return {
        "data_dir": data_dir,
        "binding_results_dir": binding_results_dir,
        "out_dir": out_dir,
        "fig_dir": fig_dir,
        "feat_dir": feat_dir,
        "feature_cache_dir": feature_cache_dir,
    }

In [4]:
for dataset in DATASETS:
    paths = get_dataset_paths(dataset)
    data_dir = paths["data_dir"]

    print(f"\nChecking {dataset}-{TASK}")
    print("Data dir:", data_dir)

    assert data_dir.exists(), f"Missing data directory: {data_dir}"

    for fname in ["test_1.csv", "test_2.csv", "test_3.csv"]:
        path = data_dir / fname
        assert path.exists(), f"Missing file: {path}"

    if not paths["binding_results_dir"].exists():
        print(f"Warning: binding results directory not found: {paths['binding_results_dir']}")
        print("Feature overlap with activity models will be skipped unless this directory exists.")
    else:
        print("Binding results directory found.")


Checking drd2-hi
Data dir: /home/f.capria/drug-discovery-lohi/data/hi/drd2
Binding results directory found.

Checking hiv-hi
Data dir: /home/f.capria/drug-discovery-lohi/data/hi/hiv
Binding results directory found.

Checking kdr-hi
Data dir: /home/f.capria/drug-discovery-lohi/data/hi/kdr
Binding results directory found.

Checking sol-hi
Data dir: /home/f.capria/drug-discovery-lohi/data/hi/sol
Binding results directory found.


## Load subsets and build pair definitions

In [5]:
SUBSET_FILES = {
    "F1": "test_3.csv",
    "F2": "test_2.csv",
    "F3": "test_1.csv",
}

PAIRS = list(combinations(["F1", "F2", "F3"], 2))


def load_subsets(dataset: str) -> dict[str, pd.DataFrame]:
    paths = get_dataset_paths(dataset)
    data_dir = paths["data_dir"]

    subsets = {}
    for subset_name, filename in SUBSET_FILES.items():
        path = data_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {path}")

        df = pd.read_csv(path).copy()
        df["subset"] = subset_name
        df["source_file"] = filename
        subsets[subset_name] = df

    return subsets


for dataset in DATASETS:
    subsets_tmp = load_subsets(dataset)
    print(f"\n{dataset.upper()}-{TASK}")
    for subset_name, df in subsets_tmp.items():
        print(f"  {subset_name} ({SUBSET_FILES[subset_name]}): {len(df)} molecules")
    print(f"  Pairs: {PAIRS}")


DRD2-hi
  F1 (test_3.csv): 1191 molecules
  F2 (test_2.csv): 1194 molecules
  F3 (test_1.csv): 1190 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]

HIV-hi
  F1 (test_3.csv): 7848 molecules
  F2 (test_2.csv): 7848 molecules
  F3 (test_1.csv): 7847 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]

KDR-hi
  F1 (test_3.csv): 2285 molecules
  F2 (test_2.csv): 3125 molecules
  F3 (test_1.csv): 3116 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]

SOL-hi
  F1 (test_3.csv): 721 molecules
  F2 (test_2.csv): 721 molecules
  F3 (test_1.csv): 721 molecules
  Pairs: [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3')]


## Featurization and pair dataset builder

In [6]:
def get_smiles_col(df: pd.DataFrame) -> str:
    """Return the SMILES column name."""
    for col in ["smiles", "SMILES", "canonical_smiles"]:
        if col in df.columns:
            return col
    raise ValueError(f"No SMILES column found. Available columns: {df.columns.tolist()}")


def get_feature_names(fp_type: str, n_features: int) -> list[str]:
    """Return feature names for each molecular representation."""
    if fp_type == "ecfp4":
        return [f"ecfp4_bit_{i}" for i in range(n_features)]

    if fp_type == "maccs":
        return [f"maccs_key_{i}" for i in range(n_features)]

    if fp_type == "rdkit_desc":
        try:
            from rdkit.Chem import Descriptors
            names = [name for name, _ in Descriptors._descList]
            return names[:n_features]
        except Exception:
            return [f"rdkit_desc_{i}" for i in range(n_features)]

    return [f"{fp_type}_feature_{i}" for i in range(n_features)]


def build_pair_dataset(
    dataset: str,
    subsets: dict[str, pd.DataFrame],
    fold_a: str,
    fold_b: str,
    fp_type: str,
) -> tuple[np.ndarray, np.ndarray, list[str], pd.DataFrame]:
    """
    Build one binary fold-discrimination dataset.

    Label convention:
        fold_a -> 0
        fold_b -> 1

    The activity label is not used here.
    """
    df_a = subsets[fold_a].copy()
    df_b = subsets[fold_b].copy()

    smiles_col = get_smiles_col(df_a)
    pair_name = f"{fold_a}_vs_{fold_b}"

    df_a["shift_label"] = 0
    df_b["shift_label"] = 1

    pair_df = pd.concat([df_a, df_b], ignore_index=True)
    pair_df["dataset"] = dataset
    pair_df["pair"] = pair_name
    pair_df["fold_a"] = fold_a
    pair_df["fold_b"] = fold_b

    smiles = pair_df[smiles_col].astype(str).tolist()
    y = pair_df["shift_label"].to_numpy(dtype=int)

    cache_path = (
        get_dataset_paths(dataset)["feature_cache_dir"]
        / f"{pair_name}_{fp_type}.npz"
    )

    X = compute_fingerprints(
        smiles_list=smiles,
        fp_type=fp_type,
        cache_path=cache_path,
    )

    feature_names = get_feature_names(fp_type, X.shape[1])

    return X, y, feature_names, pair_df

In [7]:
import os
import contextlib
from rdkit import RDLogger

VERBOSE = False

@contextlib.contextmanager
def silence_output():
    """
    Suppress noisy stdout/stderr and RDKit logs during featurization and fitting.
    """
    RDLogger.DisableLog("rdApp.*")
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield
    RDLogger.EnableLog("rdApp.*")

## In-sample discrimination — maximum separability

In [ ]:
def make_high_capacity_discriminators(fp_type: str) -> dict:
    """
    High-capacity / weakly-regularized fold discriminators.
    Used only for in-sample maximum separability.
    """
    scale = fp_type == "rdkit_desc"

    dt = DecisionTreeClassifier(
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        ccp_alpha=0.0,
        random_state=RANDOM_STATE,
    )

    lr = LogisticRegression(
        C=1e4,
        penalty="l2",
        solver="lbfgs",
        max_iter=20000,
        random_state=RANDOM_STATE,
    )

    svm = LinearSVC(
        C=1e4,
        max_iter=20000,
        random_state=RANDOM_STATE,
    )

    if scale:
        lr = Pipeline([
            ("scaler", StandardScaler()),
            ("model", lr),
        ])
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("model", svm),
        ])

    return {
        "DT": dt,
        "LR": lr,
        "SVM": svm,
    }


def shift_scores_from_balanced_accuracy(bal_acc: float) -> tuple[float, float]:
    """
    Convert balanced accuracy into:
    - shift_score_01 in [0, 1]
    - proxy_A_distance in [0, 2]
    """
    shift_score_01 = max(0.0, 2.0 * bal_acc - 1.0)
    proxy_a_distance = max(0.0, 4.0 * bal_acc - 2.0)
    return shift_score_01, proxy_a_distance


insample_rows = []
fitted_high_capacity = {}

for dataset in DATASETS:
    subsets = load_subsets(dataset)

    if VERBOSE:
        print(f"\n=== High-capacity in-sample discrimination: {dataset.upper()}-{TASK} ===")

    for fold_a, fold_b in PAIRS:
        pair_name = f"{fold_a}_vs_{fold_b}"

        for fp_type in FINGERPRINTS:
            try:
                with silence_output():
                    X, y, feature_names, pair_df = build_pair_dataset(
                        dataset=dataset,
                        subsets=subsets,
                        fold_a=fold_a,
                        fold_b=fold_b,
                        fp_type=fp_type,
                    )
            except Exception as e:
                print(f"Skip {dataset} | {pair_name} | {fp_type}: {e}")
                continue

            for model_name, model in make_high_capacity_discriminators(fp_type).items():
                if VERBOSE:
                    print(f"{dataset} | {pair_name} | {fp_type} | {model_name}")

                fitted = clone(model)

                with silence_output():
                    fitted.fit(X, y)

                y_pred = fitted.predict(X)

                acc = accuracy_score(y, y_pred)
                bal_acc = balanced_accuracy_score(y, y_pred)
                shift_score_01, proxy_a_distance = shift_scores_from_balanced_accuracy(bal_acc)

                insample_rows.append({
                    "dataset": dataset,
                    "task": TASK,
                    "pair": pair_name,
                    "fold_a": fold_a,
                    "fold_b": fold_b,
                    "fingerprint": fp_type,
                    "model": model_name,
                    "evaluation": "high_capacity_insample",
                    "accuracy": acc,
                    "balanced_accuracy": bal_acc,
                    "shift_score_01": shift_score_01,
                    "proxy_A_distance": proxy_a_distance,
                    "n_samples": len(y),
                    "n_class_0": int((y == 0).sum()),
                    "n_class_1": int((y == 1).sum()),
                })

                fitted_high_capacity[(dataset, pair_name, fp_type, model_name)] = {
                    "model": fitted,
                    "feature_names": feature_names,
                    "X": X,
                    "y": y,
                    "pair_df": pair_df,
                }

df_insample = pd.DataFrame(insample_rows)

for dataset, sub in df_insample.groupby("dataset"):
    out_path = (
        get_dataset_paths(dataset)["out_dir"]
        / "discrimination_high_capacity_insample.csv"
    )
    sub.to_csv(out_path, index=False)

df_insample.to_csv(
    OUT_ROOT / "cross_dataset_discrimination_high_capacity_insample.csv",
    index=False,
)

display(
    df_insample
    .pivot_table(
        index=["dataset", "pair", "fingerprint"],
        columns="model",
        values="balanced_accuracy",
    )
    .round(3)
)

model                            DT     LR    SVM
dataset pair     fingerprint                     
drd2    F1_vs_F2 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.958  0.952
                 rdkit_desc   1.000  0.985  0.984
        F1_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.932  0.927
                 rdkit_desc   1.000  0.963  0.962
        F2_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.974  0.972
                 rdkit_desc   1.000  0.984  0.984
hiv     F1_vs_F2 ecfp4        1.000  0.789  0.790
                 maccs        1.000  0.690  0.692
                 rdkit_desc   1.000  0.699  0.698
        F1_vs_F3 ecfp4        1.000  0.775  0.776
                 maccs        1.000  0.688  0.689
                 rdkit_desc   1.000  0.706  0.707
        F2_vs_F3 ecfp4        1.000  0.825  0.825
                 maccs        0.999  0.734  0.735
                 rdkit_desc   1.000  0.747  0.748
kdr     F1_vs_F2 ecfp4        0.816  0.784  0.783
                 maccs        0.816  0.727  0.726
                 rdkit_desc   0.816  0.730  0.734
        F1_vs_F3 ecfp4        0.817  0.784  0.784
                 maccs        0.817  0.713  0.716
                 rdkit_desc   0.817  0.734  0.734
        F2_vs_F3 ecfp4        0.683  0.683  0.683
                 maccs        0.683  0.637  0.638
                 rdkit_desc   0.683  0.648  0.651
sol     F1_vs_F2 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.651  0.655
                 rdkit_desc   1.000  0.679  0.687
        F1_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        0.999  0.714  0.716
                 rdkit_desc   1.000  0.729  0.734
        F2_vs_F3 ecfp4        1.000  1.000  1.000
                 maccs        1.000  0.693  0.692
                 rdkit_desc   1.000  0.707  0.704

- DRD2 — tutto separabile, anche linearmente. Balanced accuracy quasi 1.0 su tutte le coppie e tutti i fingerprint, inclusi LR e SVM. Significa che i tre fold di DRD2 sono chimicamente distinti non solo in modo non-lineare (quello lo fa sempre il DT) ma anche con separatori lineari su fingerprint binari. Lo shift è reale e strutturato. Però ricorda che questo è in-sample con modelli overfittati — la domanda chiave è quanto di questa separabilità sopravvive out-of-sample con la grid search.
- HIV — pattern interessante e asimmetrico. Il DT arriva sempre a 1.0 (overfit perfetto), ma LR e SVM si fermano a 0.69-0.83. Questo ti dice che i fold di HIV non sono linearmente separabili su fingerprint singoli — ci sono molecole dei due fold che occupano lo stesso spazio di feature. La coppia F2 vs F3 è la più distinguibile (LR ECFP4 = 0.825), F1 vs F2 e F1 vs F3 sono più sovrapposte. Questo suggerisce che F1 è il fold più "centrale" chimicamente, mentre F2 e F3 sono più distinti tra loro. Interessante da confrontare con i tuoi risultati OOD.
- KDR — il caso anomalo e quello più rivelatore. È l'unico dataset dove anche il DT non raggiunge 1.0. La balanced accuracy del DT è 0.816 per F1 vs F2 e F1 vs F3, e scende a 0.683 per F2 vs F3. Questo significa che i fold di KDR sono parzialmente sovrapposti anche per un classificatore con capacità illimitata — non è solo un problema di linearità ma di vera sovrapposizione nello spazio delle feature. La coppia F2 vs F3 è la meno distinguibile. Questo è paradossale rispetto a quello che ti aspettavi: KDR è il dataset dove il protocollo OOD dà i guadagni maggiori (15-35 punti), eppure i fold sono meno separabili di DRD2 e Sol. Potrebbe significare che lo shift su KDR non è di tipo "scaffold diverso" ma qualcosa di più sottile — magari i fold condividono scaffold simili ma con pattern di attività diversi, e il modello selezionato con random shuffle sovrastima la sua capacità di generalizzare proprio perché la validazione interna è fuorviante in uno spazio chimico parzialmente sovrapposto. Questa è un'ipotesi da verificare con il confronto Lista A vs Lista B.
- Sol — DT sempre a 1.0, LR e SVM collassano su MACCS e RDKit desc. Su ECFP4 LR e SVM raggiungono 1.0 — separabilità lineare perfetta su fingerprint a 1024 bit. Su MACCS e RDKit descriptors scendono a 0.65-0.73. Questo dice che lo shift in Sol è principalmente codificato nei bit ECFP4 (sottostrutture specifiche) e molto meno nei descrittori globali. È coerente col fatto che Sol-Hi è un task su solubilità, dove il scaffold fa molta differenza ma i descriptor fisico-chimici medi possono essere simili tra fold.

Il punto più importante prima dei risultati out-of-sample. KDR sfida direttamente l'ipotesi semplice "più shift → più optimism gap". Su KDR hai il gap maggiore ma la separabilità in-sample più bassa tra tutti i dataset. Questo è potenzialmente il risultato più interessante della tesi, non un problema — significa che la relazione tra shift e optimism gap è più complessa di una semplice correlazione lineare, e i due paper di Kifer e Ben-David hanno molto da dire su perché questo accade. In particolare Ben-David ti dice che il bound sulla generalizzazione dipende sia dalla A-distance sia dalla capacità del miglior classificatore congiunto — se quel classificatore congiunto è debole (fold sovrapposti), il bound è diverso da quello che ti aspetti. Vale la pena mettere da parte questa osservazione e aspettare i dati out-of-sample prima di concludere, ma tienila in testa.

## Out-of-sample discrimination — generalizable shift + A-distance

In [ ]:
from sklearn.model_selection import GridSearchCV


def allowed_fingerprints_for_model(model_name: str) -> list[str]:
    """
    Match the fingerprint availability of the main OOD-vs-random study.
    """
    if model_name == "LR":
        return ["ecfp4", "maccs", "rdkit_desc"]
    if model_name in ["DT", "SVM"]:
        return ["ecfp4", "maccs"]
    raise ValueError(f"Unknown model_name: {model_name}")


def get_main_search_grid(dataset: str, model_name: str) -> dict:
    """
    Return the same hyperparameter grid used in the main OOD-vs-random study.

    HIV uses reduced grids for computational reasons, consistently with
    the main experiments.
    """
    is_hiv = dataset == "hiv"

    if model_name == "DT":
        if is_hiv:
            return {
                "max_depth": [3, 5, 8, 10, 15],
                "min_samples_split": [10, 20, 50],
                "min_samples_leaf": [5, 10, 20, 50],
                "criterion": ["gini", "entropy"],
                "class_weight": [None, "balanced"],
                "max_features": ["sqrt", "log2"],
                "ccp_alpha": [0.0, 0.0001, 0.001],
            }

        return {
            "max_depth": [3, 5, 7, 10, 15, 20],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "criterion": ["gini", "entropy"],
            "class_weight": [None, "balanced"],
            "max_features": ["sqrt", "log2", None],
            "ccp_alpha": [0.0, 0.0001, 0.001, 0.01],
        }

    if model_name == "LR":
        if is_hiv:
            return {
                "C": [0.01, 0.1, 1.0, 5.0, 10.0],
                "l1_ratio": [0.0, 0.5, 1.0],
                "class_weight": [None, "balanced"],
            }

        return {
            "C": [0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
            "l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
            "class_weight": [None, "balanced"],
        }

    if model_name == "SVM":
        if is_hiv:
            return {
                "C": [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
                "class_weight": [None, "balanced"],
            }

        return {
            "C": [
                0.001, 0.005, 0.01, 0.05, 0.1,
                0.25, 0.5, 1.0, 2.0, 5.0,
                10.0, 25.0, 50.0, 100.0,
            ],
            "class_weight": [None, "balanced"],
        }

    raise ValueError(f"Unknown model_name: {model_name}")


def make_main_search_discriminator(dataset: str, fp_type: str, model_name: str):
    """
    Build discriminator + grid using the same search space as the main
    protocol-comparison study.
    """
    if fp_type not in allowed_fingerprints_for_model(model_name):
        return None, None

    if model_name == "DT":
        estimator = DecisionTreeClassifier(random_state=RANDOM_STATE)
        grid = get_main_search_grid(dataset, model_name)

    elif model_name == "LR":
        max_iter = 50000 if dataset == "hiv" else 15000

        base = LogisticRegression(
            solver="saga",
            penalty="elasticnet",
            max_iter=max_iter,
            random_state=RANDOM_STATE,
        )

        grid = get_main_search_grid(dataset, model_name)

        if fp_type == "rdkit_desc":
            estimator = Pipeline([
                ("scaler", StandardScaler()),
                ("model", base),
            ])
            grid = {f"model__{k}": v for k, v in grid.items()}
        else:
            estimator = base

    elif model_name == "SVM":
        estimator = LinearSVC(
            max_iter=20000,
            random_state=RANDOM_STATE,
        )
        grid = get_main_search_grid(dataset, model_name)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return estimator, grid


search_cv_rows = []
best_search_models = {}

skf = StratifiedKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for dataset in DATASETS:
    subsets = load_subsets(dataset)

    if VERBOSE:
        print(f"\n=== Same-search out-of-sample discrimination: {dataset.upper()}-{TASK} ===")

    for fold_a, fold_b in PAIRS:
        pair_name = f"{fold_a}_vs_{fold_b}"

        for fp_type in FINGERPRINTS:
            try:
                with silence_output():
                    X, y, feature_names, pair_df = build_pair_dataset(
                        dataset=dataset,
                        subsets=subsets,
                        fold_a=fold_a,
                        fold_b=fold_b,
                        fp_type=fp_type,
                    )
            except Exception as e:
                print(f"Skip {dataset} | {pair_name} | {fp_type}: {e}")
                continue

            for model_name in ["DT", "LR", "SVM"]:
                estimator, param_grid = make_main_search_discriminator(
                    dataset=dataset,
                    fp_type=fp_type,
                    model_name=model_name,
                )

                if estimator is None:
                    continue

                if VERBOSE:
                    print(f"{dataset} | {pair_name} | {fp_type} | {model_name}")

                search = GridSearchCV(
                    estimator=estimator,
                    param_grid=param_grid,
                    scoring={
                        "accuracy": "accuracy",
                        "balanced_accuracy": "balanced_accuracy",
                        "roc_auc": "roc_auc",
                    },
                    refit="balanced_accuracy",
                    cv=skf,
                    n_jobs=-1,
                    return_train_score=True,
                    error_score=np.nan,
                )

                with silence_output():
                    search.fit(X, y)

                best_idx = search.best_index_
                cvres = search.cv_results_

                acc_mean = float(cvres["mean_test_accuracy"][best_idx])
                acc_std = float(cvres["std_test_accuracy"][best_idx])

                bal_mean = float(cvres["mean_test_balanced_accuracy"][best_idx])
                bal_std = float(cvres["std_test_balanced_accuracy"][best_idx])

                roc_mean = float(cvres["mean_test_roc_auc"][best_idx])
                roc_std = float(cvres["std_test_roc_auc"][best_idx])

                train_bal_mean = float(cvres["mean_train_balanced_accuracy"][best_idx])
                train_bal_std = float(cvres["std_train_balanced_accuracy"][best_idx])

                shift_score_01, proxy_a_distance = shift_scores_from_balanced_accuracy(
                    bal_mean
                )

                search_cv_rows.append({
                    "dataset": dataset,
                    "task": TASK,
                    "pair": pair_name,
                    "fold_a": fold_a,
                    "fold_b": fold_b,
                    "fingerprint": fp_type,
                    "model": model_name,
                    "evaluation": "same_search_cv",
                    "cv_accuracy_mean": acc_mean,
                    "cv_accuracy_std": acc_std,
                    "cv_balanced_accuracy_mean": bal_mean,
                    "cv_balanced_accuracy_std": bal_std,
                    "cv_roc_auc_mean": roc_mean,
                    "cv_roc_auc_std": roc_std,
                    "cv_train_balanced_accuracy_mean": train_bal_mean,
                    "cv_train_balanced_accuracy_std": train_bal_std,
                    "cv_train_test_gap_balanced_accuracy": train_bal_mean - bal_mean,
                    "shift_score_01": shift_score_01,
                    "proxy_A_distance": proxy_a_distance,
                    "best_params": json.dumps(search.best_params_),
                    "n_samples": len(y),
                    "n_class_0": int((y == 0).sum()),
                    "n_class_1": int((y == 1).sum()),
                })

                best_search_models[(dataset, pair_name, fp_type, model_name)] = {
                    "model": search.best_estimator_,
                    "feature_names": feature_names,
                    "X": X,
                    "y": y,
                    "pair_df": pair_df,
                    "best_params": search.best_params_,
                }

df_search_cv = pd.DataFrame(search_cv_rows)

for dataset, sub in df_search_cv.groupby("dataset"):
    out_path = (
        get_dataset_paths(dataset)["out_dir"]
        / "discrimination_same_search_cv.csv"
    )
    sub.to_csv(out_path, index=False)

df_search_cv.to_csv(
    OUT_ROOT / "cross_dataset_discrimination_same_search_cv.csv",
    index=False,
)

display(
    df_search_cv
    .pivot_table(
        index=["dataset", "pair", "fingerprint"],
        columns="model",
        values="proxy_A_distance",
    )
    .round(3)
)